# 00 — Setup lingkungan KulitKu di Google Colab
Jalankan sel berurutan. Notebook ini hanya menyiapkan lingkungan: belum cleaning, split, ekstraksi fitur atau training.

**Sebelum mulai:** pilih Runtime → Change runtime type → GPU jika tersedia. Akses GPU Colab gratis tidak dijamin. Siapkan repository GitHub dan dataset di Drive. Dataset sumber tidak diubah. Runtime lokal Colab bersifat sementara.

Reproducibility di sini berarti kode, seed, paket dan perangkat tercatat. Hasil identik lintas versi/perangkat tidak dijamin. Jangan menaruh token di URL atau sel notebook.


## 1. Hubungkan Google Drive


In [4]:
from google.colab import drive
from pathlib import Path
import os, sys, json, subprocess, platform, hashlib, shutil
from datetime import datetime, timezone

drive.mount('/content/drive')


Mounted at /content/drive


## 2. Isi konfigurasi
Ganti REPO_URL. Ganti DRIVE_ROOT jika folder KulitKu berada di lokasi berbeda. DATA_SOURCE harus menunjuk langsung ke folder berisi empat folder kelas, bukan ZIP.

REPO_REF boleh berupa branch untuk percobaan pertama. Setelah clone, SHA commit dicatat; gunakan SHA itu pada eksperimen berikutnya. Untuk repo private, masukkan Personal Access Token melalui ikon kunci **Colab Secrets**, nama `GITHUB_TOKEN`, dan aktifkan akses notebook. Token tidak disimpan ke file atau remote URL.


In [5]:
REPO_URL = 'https://github.com/Liex2810/KulitKu.git'
REPO_REF = 'main'  # gunakan commit SHA untuk pengulangan eksperimen
PRIVATE_REPO = False
DRIVE_ROOT = Path('/content/drive/MyDrive/KulitKu')
DATA_SOURCE = DRIVE_ROOT / 'datasets' / 'kulitku_4kelas.zip'  # boleh juga folder raw
CLASSES = ['Eczema', 'Herpes Zoster', 'Normal', 'Ringworm']
SEED = 42
REPO_LOCAL = Path('/content/kulitku-repo')
DATA_LOCAL = Path('/content/kulitku-data/raw')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
RUN_LOCAL = Path('/content/kulitku-runs') / RUN_ID
RUN_DRIVE = DRIVE_ROOT / 'experiments' / 'setup' / RUN_ID
RUN_LOCAL.mkdir(parents=True, exist_ok=True)
RUN_DRIVE.mkdir(parents=True, exist_ok=True)
print('Run:', RUN_ID)
print('Dataset sumber:', DATA_SOURCE)


Run: 20261105T070903_159197Z
Dataset sumber: /content/drive/MyDrive/KulitKu/datasets/kulitku_4kelas.zip


## 3. Clone repository dan kunci commit
Checkout bersih diperlukan agar identitas kode tidak ambigu. Sel berhenti jika direktori tujuan sudah ada, untuk menghindari menimpa pekerjaan.


In [6]:
from urllib.parse import urlparse
parsed = urlparse(REPO_URL)
if parsed.scheme != 'https' or parsed.hostname != 'github.com' or parsed.username or parsed.password:
    raise ValueError('Gunakan URL HTTPS github.com tanpa kredensial.')
if 'USERNAME' in REPO_URL:
    raise ValueError('Isi REPO_URL dengan repository milikmu dahulu.')
if REPO_LOCAL.exists():
    raise FileExistsError('Folder repo sudah ada. Jika clone sebelumnya berhasil, lanjut sel berikutnya; jangan menimpa perubahan.')

def git_call(args, **kwargs):
    result = subprocess.run(['git', *args], capture_output=True, text=True, **kwargs)
    if result.returncode:
        raise RuntimeError('Operasi Git gagal. Periksa URL, ref, koneksi dan izin token. Detail sensitif tidak dicetak.')
    return result.stdout.strip()

askpass = Path('/content/kulitku-git-askpass.py')
git_env = os.environ.copy()
try:
    if PRIVATE_REPO:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')
        if not token:
            raise ValueError('Secret GITHUB_TOKEN belum diisi.')
        askpass.write_text('#!/usr/bin/env python3\nimport os, sys\nprompt = sys.argv[1].lower()\nprint("x-access-token" if "username" in prompt else os.environ["KULITKU_GIT_TOKEN"])\n')
        askpass.chmod(0o700)
        git_env.update(GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT='0', KULITKU_GIT_TOKEN=token)
    else:
        git_env['GIT_TERMINAL_PROMPT'] = '0'
    git_call(['clone', '--no-checkout', REPO_URL, str(REPO_LOCAL)], env=git_env)
    git_call(['-C', str(REPO_LOCAL), 'checkout', '--detach', REPO_REF], env=git_env)
finally:
    askpass.unlink(missing_ok=True)
    git_env.pop('KULITKU_GIT_TOKEN', None)
    if 'token' in globals():
        del token

commit = git_call(['-C', str(REPO_LOCAL), 'rev-parse', 'HEAD'])
print('Commit terkunci:', commit)
print('Repo lokal:', REPO_LOCAL)


Commit terkunci: 9aff0d28a1946fc06c29a7ceff999ac1ac494772
Repo lokal: /content/kulitku-repo


## 4. Instal dependensi setup/EDA/ekstraksi fitur
Paket tambahan di bawah dipasang dengan versi eksplisit. Torch, torchvision, NumPy dan SciPy mengikuti runtime Colab yang tersedia, kemudian seluruh versi aktual direkam. Jangan mengganti pasangan torch/torchvision CUDA secara sembarang.

Ini baseline setup baru, bukan klaim kompatibilitas dengan semua runtime Colab atau dependensi backend repository. Jangan menjalankan pip install -e . secara otomatis: backend/Flutter bukan kebutuhan notebook ini. Jika kode ML repo membutuhkan tambahan, deklarasikan dan simpan lock baru sebelum eksperimen.

Jika Colab meminta restart setelah instalasi, restart session lalu jalankan sel 1–2, lewati clone yang sudah berhasil, dan lanjut sel 5. Jangan mengabaikan kegagalan import atau pip check.


In [10]:
%pip install jedi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 25.2 MB/s  0:00:00


In [7]:
PACKAGES = ['timm==1.0.22', 'ImageHash==4.3.2']
subprocess.run([sys.executable, '-m', 'pip', 'install', *PACKAGES], check=True)
(RUN_LOCAL / 'requirements-bootstrap.txt').write_text('\n'.join(PACKAGES) + '\n')
print('Instalasi selesai. Jika Colab meminta restart, lakukan sebelum melanjutkan.')


Instalasi selesai. Jika Colab meminta restart, lakukan sebelum melanjutkan.


## 5. Periksa import, paket dan GPU


In [11]:
import importlib
import importlib.metadata as metadata
import random

required_modules = ['numpy', 'pandas', 'torch', 'torchvision', 'timm',
                    'sklearn', 'PIL', 'matplotlib', 'seaborn', 'tqdm', 'imagehash']
import_errors = {}
for module_name in required_modules:
    try:
        globals()[module_name] = importlib.import_module(module_name)
    except Exception as exc:
        import_errors[module_name] = f'{type(exc).__name__}: {exc}'
if import_errors:
    print(json.dumps(import_errors, indent=2))
    raise RuntimeError(
        'Import gagal. Jika baru menginstal paket, restart session lalu jalankan bagian 1–2 dan 5. '
        'Jika masih gagal, gunakan runtime baru dan sertakan laporan import di atas.'
    )
np = numpy
pd = pandas
check = subprocess.run([sys.executable, '-m', 'pip', 'check'],
                       capture_output=True, text=True)
(RUN_LOCAL / 'pip-check.txt').write_text(check.stdout + check.stderr)
print(check.stdout + check.stderr)
DEPENDENCIES_OK = check.returncode == 0
print('GPU tersedia:', torch.cuda.is_available())
print('Torch:', torch.__version__, '| torchvision:', torchvision.__version__)
# Uji operasi torchvision yang sering gagal ketika pasangan wheel tidak cocok.
from torchvision.ops import nms
nms(torch.tensor([[0., 0., 1., 1.]]), torch.tensor([0.9]), 0.5)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('CPU aktif. Setup/EDA tetap dapat berjalan.')
if not DEPENDENCIES_OK:
    # Simpan laporan ke Drive sebelum menghentikan proses.
    shutil.copy2(RUN_LOCAL / 'pip-check.txt', RUN_DRIVE / 'pip-check.txt')
    raise RuntimeError(
        'Ada konflik dependensi; detail ada di output dan pip-check.txt di folder run Drive. '
        'Gunakan runtime baru lalu jalankan notebook ini berurutan. '
        'Jika konflik tetap ada, perbaikan perlu mengikuti nama paket dan versi pada laporan; '
        'jangan lanjut training atau mengabaikan pip check.'
    )


No broken requirements found.

GPU tersedia: True
Torch: 2.11.0+cu130 | torchvision: 0.26.0+cu130
GPU: Tesla T4


## 6. Tetapkan seed dan determinisme
Seed bukan jaminan hasil identik lintas perangkat/versi. Setting deterministik yang tidak didukung akan menghasilkan error, agar perbedaan tidak disembunyikan. PYTHONHASHSEED harus diatur sebelum proses Python dimulai untuk mengendalikan hash; pengaturan dalam sel hanya diwariskan ke subprocess berikutnya. Gunakan urutan file yang disortir.


In [12]:
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['PYTHONHASHSEED'] = str(SEED)  # efektif untuk proses Python baru
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True)

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

loader_generator = torch.Generator().manual_seed(SEED)
# DataLoader nanti: worker_init_fn=seed_worker, generator=loader_generator
# Classifier/split nanti tetap perlu random_state=SEED.
print('Seed:', SEED, '| mode deterministik aktif')


Seed: 42 | mode deterministik aktif


## 7. Salin dataset dari Drive ke disk lokal Colab
Sel ini menyalin seluruh isi empat folder kelas tanpa cleaning. Tanda selesai ditulis hanya setelah penyalinan dan pemeriksaan ukuran/jumlah file berhasil. Jika data sumber berubah, gunakan folder lokal baru; cache bukan sistem sinkronisasi. Perbandingan ukuran file bukan verifikasi isi bit-per-bit; hash lengkap dilakukan di tahap inventarisasi.

Membaca ribuan file di Drive masih dapat memerlukan waktu. Kelak arsip dataset versi tetap dapat mempercepat staging; notebook ini memakai struktur folder yang sudah kamu miliki.


In [14]:
from pathlib import Path

root = Path('/content/drive/MyDrive/KulitKu')
candidates = [
    p.parent for p in root.rglob('Eczema')
    if p.is_dir() and all((p.parent / c).is_dir() for c in CLASSES)
]

if len(candidates) != 1:
    raise ValueError(
        f'Ditemukan {len(candidates)} folder dataset: {candidates}. '
        'Pastikan ZIP sudah diekstrak dan lokasi root sesuai.'
    )

DATA_SOURCE = candidates[0]
print('DATA_SOURCE:', DATA_SOURCE)

DATA_SOURCE: /content/drive/MyDrive/KulitKu/datasets/original


In [18]:
from pathlib import Path
import shutil
import zipfile
from tqdm.auto import tqdm

ZIP_DRIVE = Path(
    '/content/drive/MyDrive/KulitKu/datasets/kulitku_4kelas.zip'
)
ZIP_LOCAL = Path('/content/kulitku_4kelas.zip')
EXTRACT_DIR = Path('/content/kulitku-zip-data')

if not ZIP_DRIVE.is_file():
    raise FileNotFoundError(f'ZIP tidak ditemukan: {ZIP_DRIVE}')

# Salin satu ZIP dengan progres berdasarkan ukuran byte.
with ZIP_DRIVE.open('rb') as src, ZIP_LOCAL.open('wb') as dst:
    with tqdm(
        total=ZIP_DRIVE.stat().st_size,
        desc='Menyalin ZIP',
        unit='B', unit_scale=True
    ) as bar:
        while True:
            block = src.read(8 * 1024 * 1024)
            if not block:
                break
            dst.write(block)
            bar.update(len(block))

# Ekstrak dari disk lokal.
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ZIP_LOCAL) as archive:
    members = archive.infolist()

    for info in members:
        target = (EXTRACT_DIR / info.filename).resolve()
        if not target.is_relative_to(EXTRACT_DIR.resolve()):
            raise ValueError(f'Path ZIP tidak aman: {info.filename}')

    for info in tqdm(members, desc='Mengekstrak', unit='file'):
        archive.extract(info, EXTRACT_DIR)

DATA_LOCAL = EXTRACT_DIR / 'raw'
DATA_ROOT = DATA_LOCAL

missing = [c for c in CLASSES if not (DATA_ROOT / c).is_dir()]
if missing:
    raise FileNotFoundError(f'Folder kelas belum ditemukan: {missing}')

print('Dataset siap:', DATA_ROOT)

Menyalin ZIP:   0%|          | 0.00/794M [00:00<?, ?B/s]

Mengekstrak:   0%|          | 0/30690 [00:00<?, ?file/s]

FileNotFoundError: Folder kelas belum ditemukan: ['Eczema', 'Herpes Zoster', 'Normal', 'Ringworm']

In [19]:
from pathlib import Path

base = Path('/content/kulitku-zip-data')
candidates = [
    p for p in [base, *base.rglob('*')]
    if p.is_dir() and all((p / c).is_dir() for c in CLASSES)
]

if len(candidates) != 1:
    print('Folder tingkat atas:', [p.name for p in base.iterdir()])
    raise ValueError(f'Ditemukan {len(candidates)} lokasi kelas: {candidates}')

DATA_LOCAL = DATA_ROOT = candidates[0]
print('Dataset siap:', DATA_ROOT)

Dataset siap: /content/kulitku-zip-data/kulitku_4kelas/raw


## 8. Simpan konfigurasi, versi lingkungan dan identitas kode
Snapshot pip freeze mencatat versi aktual; ini bukan lock lintas OS/CUDA yang dijamin portabel. Untuk pengulangan, cocokkan Python, torch/CUDA, dan perangkat sebelum memulihkan paket. Jangan mengunggah pip freeze mentah tanpa pemeriksaan: kode di bawah meredaksi URL dependency langsung yang mungkin memuat kredensial.


In [20]:
import re
commit = git_call(['-C', str(REPO_LOCAL), 'rev-parse', 'HEAD'])
dirty = git_call(['-C', str(REPO_LOCAL), 'status', '--porcelain'])
if dirty:
    raise RuntimeError('Repository berubah dari commit. Commit perubahan atau catat patch terpisah sebelum eksperimen.')
versions = {name: metadata.version(name) for name in [
    'torch','torchvision','timm','numpy','scipy','scikit-learn','pandas',
    'Pillow','matplotlib','seaborn','tqdm','ImageHash']}
gpu_info = []
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        gpu_info.append({'name':props.name,'memory_bytes':props.total_memory})
environment = {
    'run_id':RUN_ID, 'timestamp_utc':datetime.now(timezone.utc).isoformat(),
    'python':sys.version, 'platform':platform.platform(), 'machine':platform.machine(),
    'cpu_count':os.cpu_count(), 'cpu_model':platform.processor(),
    'packages':versions, 'cuda_available':torch.cuda.is_available(),
    'torch_cuda':torch.version.cuda, 'cudnn':torch.backends.cudnn.version(),
    'gpu':gpu_info, 'git_commit':commit, 'git_dirty':False,
    'seed':SEED, 'deterministic_algorithms':True,
    'pythonhashseed_current_process_guaranteed':False,
    'dataset_content_hash_verified':False,
    'audit_completed':False, 'training_performed':False
}
config = {'repo_url':REPO_URL,'requested_ref':REPO_REF,'commit':commit,
    'drive_root':str(DRIVE_ROOT),'dataset_source':str(DATA_SOURCE),
    'dataset_local':str(DATA_ROOT),'classes':CLASSES,'seed':SEED}
(RUN_LOCAL/'environment.json').write_text(json.dumps(environment, indent=2))
(RUN_LOCAL/'setup-config.json').write_text(json.dumps(config, indent=2))
(RUN_LOCAL/'source-size-manifest.json').write_text(json.dumps(source_sizes, sort_keys=True))
freeze = subprocess.run([sys.executable,'-m','pip','freeze'],check=True,capture_output=True,text=True).stdout
safe_freeze = []
for line in freeze.splitlines():
    if ' @ ' in line or '://' in line or line.startswith('-e '):
        safe_freeze.append('# Direct/local dependency omitted; inspect privately and pin a public version if needed.')
    else:
        safe_freeze.append(line)
(RUN_LOCAL/'requirements-snapshot.txt').write_text('\n'.join(safe_freeze)+'\n')
if Path('/proc/cpuinfo').exists():
    (RUN_LOCAL/'cpu-info.txt').write_text(Path('/proc/cpuinfo').read_text())
try:
    gpu_cmd = subprocess.run(['nvidia-smi'],capture_output=True,text=True)
    (RUN_LOCAL/'nvidia-smi.txt').write_text(gpu_cmd.stdout+gpu_cmd.stderr)
except FileNotFoundError:
    (RUN_LOCAL/'nvidia-smi.txt').write_text('nvidia-smi tidak tersedia; kemungkinan runtime CPU.')
print(json.dumps({'commit':commit,'packages':versions,'GPU':gpu_info},indent=2))


{
  "commit": "9aff0d28a1946fc06c29a7ceff999ac1ac494772",
  "packages": {
    "torch": "2.11.0+cu130",
    "torchvision": "0.26.0+cu130",
    "timm": "1.0.22",
    "numpy": "2.4.6",
    "scipy": "1.16.3",
    "scikit-learn": "1.6.1",
    "pandas": "2.2.3",
    "Pillow": "11.3.0",
    "matplotlib": "3.10.0",
    "seaborn": "0.13.2",
    "tqdm": "4.67.3",
    "ImageHash": "4.3.2"
  },
  "GPU": [
    {
      "name": "Tesla T4",
      "memory_bytes": 15637086208
    }
  ]
}


## 9. Backup hasil penting ke Drive
Panggil fungsi ini lagi setelah membuat hasil baru. Fungsi hanya menyalin folder hasil run ini, bukan dataset atau repository. Tidak ada sinkronisasi latar belakang otomatis; panggil di akhir tiap tahap/kombo eksperimen. Model besar disimpan per tahap, bukan setiap batch.


In [21]:
def backup_run():
    copied = []
    for src in sorted(RUN_LOCAL.rglob('*')):
        if not src.is_file():
            continue
        relative = src.relative_to(RUN_LOCAL)
        dest = RUN_DRIVE / relative
        dest.parent.mkdir(parents=True, exist_ok=True)
        temporary = dest.with_name(dest.name + '.partial')
        shutil.copy2(src, temporary)
        if src.stat().st_size != temporary.stat().st_size:
            raise IOError(f'Ukuran backup tidak cocok: {relative}')
        os.replace(temporary, dest)
        copied.append(str(relative))
    print(f'Backup {len(copied)} berkas:', RUN_DRIVE)
    return copied

backup_run()
print('SETUP SELESAI. Lanjut notebook inventarisasi/EDA; belum boleh split atau training sebelum audit.')


Backup 8 berkas: /content/drive/MyDrive/KulitKu/experiments/setup/20261105T070903_159197Z
SETUP SELESAI. Lanjut notebook inventarisasi/EDA; belum boleh split atau training sebelum audit.


## Output yang diharapkan
Di `Drive/KulitKu/experiments/setup/<run_id>/` tersedia environment.json, setup-config.json, requirements-bootstrap.txt, requirements-snapshot.txt, pip-check.txt, source-size-manifest.json, cpu-info.txt dan nvidia-smi.txt.

Untuk mengulangi: gunakan commit tercatat, periksa versi runtime terhadap environment.json, pulihkan paket pada lingkungan kompatibel, jalankan setup dan bandingkan versi aktual. Tidak ada jaminan runtime gratis berikutnya menyediakan GPU yang sama.

Untuk GitHub, simpan notebook ini di `machine-learning/notebooks/00_setup_colab.ipynb`. Jangan commit data atau kredensial. File lingkungan boleh masuk laporan setelah ditinjau.

**Referensi:** [Colab FAQ](https://research.google.com/colaboratory/faq.html), [Colab I/O](https://colab.research.google.com/notebooks/io.ipynb), [PyTorch reproducibility](https://docs.pytorch.org/docs/stable/notes/randomness.html).
